# Przypisanie PS do budynków

Celem jest przygotowanie warstwy budynków, którą będzie można wykorzystać do dalszych analiz przestrzennych i wskazywania obiektów wymagających dokładniejszej obserwacji.

## Wczytanie danych

- EGMS_Turow_U_results.csv – punkty EGMS z cechami przemieszczeń oraz wynikami Isolation Forest i K-means (wynik notatnika `EGMS_analysis_pipeline.ipynb`);

- budynki_Turow.gpkg – obrysy budynków EUBUCCO z Czech, Polski i Niemiec (przygotowane w notatniku `EGMS_data_preparation.ipynb`).

In [34]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

ps = pd.read_csv("EGMS_Turow_U_E_results.csv")
buildings = gpd.read_file("budynki_Turow.gpkg")

print("Liczba PS:", len(ps))
print("Liczba budynków:", len(buildings))
print("CRS budynków:", buildings.crs)

Liczba PS: 6268
Liczba budynków: 39156
CRS budynków: EPSG:3035


## Usunięcie powtarzających się geometrii

In [35]:
# Usunięcie identycznych geometrii
buildings = (
    buildings
    .drop_duplicates(subset="geometry")
    .reset_index(drop=True)
    .copy()
)

# Nadanie identyfikatorów budynkom
buildings["building_id"] = np.arange(
    1, len(buildings) + 1
)

print("Liczba budynków:", len(buildings))

Liczba budynków: 39156


## Ujednolicenie układów współrzędnych

In [36]:
ps_gdf = gpd.GeoDataFrame(
    ps.copy(),
    geometry=gpd.points_from_xy(
        ps["easting"],
        ps["northing"]
    ),
    crs="EPSG:3035"
)

# Transformacja budynków do układu punktów EGMS
buildings = buildings.to_crs(ps_gdf.crs)

print("CRS PS:", ps_gdf.crs)
print("CRS budynków:", buildings.crs)

CRS PS: EPSG:3035
CRS budynków: EPSG:3035


## Wybór geometrii i atrybutów

In [37]:
building_footprints = buildings[
    ["building_id", "geometry"]
].copy()

print("Typy geometrii:")
print(building_footprints.geometry.geom_type.value_counts())

Typy geometrii:
Polygon    39156
Name: count, dtype: int64


In [38]:
ps_for_join = ps_gdf[
    [
        "pid",
        "mean_velocity",
        "acceleration",
        "trend_change",
        "anomaly_score",
        "anomaly",
        "cluster",
        "rmse_ts",
        "geometry"
    ]
].copy()

## Przypisanie najbliższego PS

Każdemu budynkowi przypisujemy najbliższy punkt PS, jeżeli leży on nie dalej niż `MAX_DISTANCE` od obrysu budynku.

Przyjmujemy **30 m**, zgodnie z dalszą częścią analizy (kolumna `has_PS_30m`). Punkty EGMS L3 są środkami komórek siatki 100 × 100 m, więc próg 30 m oznacza, że budynkowi przypisujemy tylko punkt z jego bezpośredniego sąsiedztwa, a budynki położone dalej pozostają bez PS. Wpływ przyjętego progu na wynik można ocenić w analizie wrażliwości.

In [39]:
MAX_DISTANCE = 30  # [m]

buildings_ps_all = gpd.sjoin_nearest(
    building_footprints,
    ps_for_join,
    how="left",
    max_distance=MAX_DISTANCE,
    distance_col="distance_PS"
)

In [40]:
buildings_ps = (
    buildings_ps_all
    .reset_index(drop=True)
    .sort_values(
        ["building_id", "distance_PS", "anomaly_score"],
        ascending=[True, True, False],
        na_position="last"
    )
    .drop_duplicates(
        subset="building_id",
        keep="first"
    )
    .reset_index(drop=True)
)

buildings_ps["has_PS_30m"] = (
    buildings_ps["pid"].notna()
)

In [41]:
matched = buildings_ps["has_PS_30m"]

print("Wszystkie budynki:", len(buildings_ps))
print("Budynki z PS do 30 m:", matched.sum())
print("Budynki bez PS do 30 m:", (~matched).sum())
print("Wykorzystane unikalne PS:", buildings_ps["pid"].nunique())
print(
    "PS wewnątrz budynku:",
    (buildings_ps["distance_PS"] == 0).sum()
)
print(
    "Mediana odległości:",
    round(
        buildings_ps.loc[
            matched,
            "distance_PS"
        ].median(),
        1
    ),
    "m"
)

Wszystkie budynki: 39156
Budynki z PS do 30 m: 15376
Budynki bez PS do 30 m: 23780
Wykorzystane unikalne PS: 3450
PS wewnątrz budynku: 438
Mediana odległości: 19.4 m


## Utworzenie warstwy wynikowej

In [42]:
result = buildings.merge(
    buildings_ps.drop(
        columns=["geometry", "index_right"],
        errors="ignore"
    ),
    on="building_id",
    how="left"
)

result = gpd.GeoDataFrame(
    result,
    geometry="geometry",
    crs=buildings.crs
)

print("Budynki w wyniku:", len(result))
print("Budynki z przypisanym PS:", result["has_PS_30m"].sum())
print("Unikalne identyfikatory:", result["building_id"].nunique())

Budynki w wyniku: 39156
Budynki z przypisanym PS: 15376
Unikalne identyfikatory: 39156


In [43]:
result.to_file(
    "budynki_EGMS_results_U_E.gpkg",
    layer="budynki_EGMS",
    driver="GPKG"
)

print("Zapisano: budynki_EGMS_results_U_E.gpkg")

Zapisano: budynki_EGMS_results_U_E.gpkg
